# Dados e EDA — Redes Neurais (TrashNet)

Integrante 1. Não precisa de GPU.

Este notebook baixa o TrashNet, roda o `01_eda_split.py` e confere que a divisão gerada é **idêntica** ao `split.json` do repositório (o mesmo usado por todos os experimentos). A execução grava numa pasta temporária, então os arquivos do repositório não são alterados.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
%cd /content/drive/MyDrive
!test -d neural-network-project || git clone https://github.com/matheusvsmp/neural-network-project.git
%cd neural-network-project
!git pull
!ls

In [ ]:
# Baixa o TrashNet (só na primeira vez)
!test -d dataset-resized || (git clone --depth 1 https://github.com/garythung/trashnet.git /content/trashnet && unzip -q /content/trashnet/data/dataset-resized.zip -d . && rm -rf __MACOSX)
!ls dataset-resized

## 1. Integridade, contagens e divisão

Verifica cada imagem (abre, confere tamanho e modo de cor), procura duplicatas por hash MD5, faz o split estratificado 70/15/15 com semente 42 (com as duplicatas fixadas no treino) e calcula os pesos de classe só no treino.

In [ ]:
!mkdir -p /content/eda_check
!python 01_eda_split.py --data_dir dataset-resized --out_json /content/eda_check/split.json --fig_dir /content/eda_check/figuras

## 2. Reprodutibilidade

Compara a divisão gerada agora com o `split.json` do repositório.

In [ ]:
import json
oficial = json.load(open('split.json', encoding='utf-8'))
novo = json.load(open('/content/eda_check/split.json', encoding='utf-8'))
for k in ['train', 'val', 'test', 'class_weights']:
    print(f'{k:14s}', 'idêntico' if oficial[k] == novo[k] else 'DIFERENTE')
assert all(oficial[k] == novo[k] for k in ['train', 'val', 'test', 'class_weights'])
print('\nO split.json do repositório é reproduzido exatamente.')

## 3. Distribuição por classe e conjunto

In [ ]:
import pandas as pd
classes = oficial['classes']
tab = pd.DataFrame({c: oficial['counts'][c] for c in ['train', 'val', 'test']}).loc[classes]
tab.columns = ['Treino', 'Validação', 'Teste']
tab['Total'] = tab.sum(axis=1)
tab['% do total'] = (100 * tab['Total'] / tab['Total'].sum()).round(1)
tab['Peso de classe (treino)'] = [oficial['class_weights'][c] for c in classes]
tab.loc['Total'] = [tab['Treino'].sum(), tab['Validação'].sum(), tab['Teste'].sum(), tab['Total'].sum(), 100.0, None]
tab[['Treino', 'Validação', 'Teste', 'Total']] = tab[['Treino', 'Validação', 'Teste', 'Total']].astype(int)
tab.fillna('')

## 4. Figuras da EDA (as do repositório, usadas no artigo)

In [ ]:
from IPython.display import Image, display
for f in ['amostras_por_classe', 'distribuicao_classes', 'distribuicao_split',
          'mesma_imagem_tres_resolucoes', 'duplicatas', 'cor_media_por_classe']:
    print(f)
    display(Image(f'figuras/eda/{f}.png'))

## 5. Resumo

- 2.527 imagens, nenhuma corrompida, todas 512 × 384 RGB.
- Desbalanceamento: paper 594 (23,5%) … trash 137 (5,4%); razão maior/menor = 4,34. Por isso o F1 macro é a métrica principal e os pesos de classe são testados.
- 3 pares de imagens idênticas com rótulos diferentes (glass115 = metal91, glass176 = plastic152, glass389 = plastic332): mantidos e fixados no treino, sem vazamento para validação e teste.
- Divisão estratificada com semente 42: treino 1.769, validação 379, teste 379. Pesos de classe só no treino (trash 3,07; paper 0,71).
- Em 128 × 96, detalhes como dobras e reflexos do metal ficam borrados (Figura `mesma_imagem_tres_resolucoes`), o que motiva a comparação de resolução.